In [1]:
import pandas as pd
import psycopg2
from dotenv import load_dotenv
import os

load_dotenv()
conn = psycopg2.connect(os.environ["DATABASE_URL"])
df = pd.read_sql("SELECT * FROM books_clean;", conn)
df.shape

C:\Users\QUEEN\AppData\Local\Temp\ipykernel_4664\997836846.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM books_clean;", conn)


(1940, 9)

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
from sklearn.preprocessing import OneHotEncoder
import numpy as np

ml_df = df[df["avg_rating"].notna()].copy()

ml_df["page_count"] = ml_df["page_count"].fillna(ml_df["page_count"].median())
ml_df["primary_category"] = ml_df["primary_category"].fillna("Unknown")
ml_df["published_year"] = ml_df["published_year"].fillna(ml_df["published_year"].median())
ml_df["ratings_count"] = ml_df["ratings_count"].fillna(0)

print(ml_df.shape)

(183, 9)


In [3]:
features = ["page_count", "published_year", "ratings_count", "primary_category"]
target = "avg_rating"

X = pd.get_dummies(ml_df[features], columns=["primary_category"], drop_first=True)
y = ml_df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (146, 48) Test: (37, 48)


In [4]:
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)

lr_mae = mean_absolute_error(y_test, lr_preds)
lr_rmse = mean_squared_error(y_test, lr_preds) ** 0.5
lr_r2 = r2_score(y_test, lr_preds)

print("Linear Regression:")
print(f"  MAE: {lr_mae:.3f}")
print(f"  RMSE: {lr_rmse:.3f}")
print(f"  R^2: {lr_r2:.3f}")

Linear Regression:
  MAE: 0.939
  RMSE: 1.227
  R^2: -0.205


In [5]:
rf = RandomForestRegressor(max_depth=6, n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_preds)
rf_rmse = mean_squared_error(y_test, rf_preds) ** 0.5
rf_r2 = r2_score(y_test, rf_preds)

print("Random Forest Regressor (max_depth=6, n_estimators=200):")
print(f"  MAE: {rf_mae:.3f}")
print(f"  RMSE: {rf_rmse:.3f}")
print(f"  R^2: {rf_r2:.3f}")

Random Forest Regressor (max_depth=6, n_estimators=200):
  MAE: 0.911
  RMSE: 1.216
  R^2: -0.182


In [6]:
# Keep only the top 8 categories, group the rest as "Other"
top_categories = ml_df["primary_category"].value_counts().head(8).index
ml_df["category_grouped"] = ml_df["primary_category"].where(
    ml_df["primary_category"].isin(top_categories), "Other"
)

features = ["page_count", "published_year", "ratings_count", "category_grouped"]
target = "avg_rating"

X = pd.get_dummies(ml_df[features], columns=["category_grouped"], drop_first=True)
y = ml_df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (146, 11) Test: (37, 11)


In [7]:
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)

lr_mae = mean_absolute_error(y_test, lr_preds)
lr_rmse = mean_squared_error(y_test, lr_preds) ** 0.5
lr_r2 = r2_score(y_test, lr_preds)

print("Linear Regression:")
print(f"  MAE: {lr_mae:.3f}")
print(f"  RMSE: {lr_rmse:.3f}")
print(f"  R^2: {lr_r2:.3f}")

Linear Regression:
  MAE: 0.935
  RMSE: 1.181
  R^2: -0.116


In [8]:
rf = RandomForestRegressor(max_depth=6, n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_preds)
rf_rmse = mean_squared_error(y_test, rf_preds) ** 0.5
rf_r2 = r2_score(y_test, rf_preds)

print("Random Forest Regressor (max_depth=6, n_estimators=200):")
print(f"  MAE: {rf_mae:.3f}")
print(f"  RMSE: {rf_rmse:.3f}")
print(f"  R^2: {rf_r2:.3f}")

Random Forest Regressor (max_depth=6, n_estimators=200):
  MAE: 0.947
  RMSE: 1.240
  R^2: -0.229


In [9]:
print("Overfitting check (Random Forest, different max_depth values):\n")
for depth in [2, 6, None]:
    model = RandomForestRegressor(max_depth=depth, n_estimators=200, random_state=42)
    model.fit(X_train, y_train)
    train_r2 = r2_score(y_train, model.predict(X_train))
    test_r2 = r2_score(y_test, model.predict(X_test))
    print(f"  max_depth={depth}: train R^2={train_r2:.3f}, test R^2={test_r2:.3f}")

Overfitting check (Random Forest, different max_depth values):

  max_depth=2: train R^2=0.193, test R^2=-0.118
  max_depth=6: train R^2=0.620, test R^2=-0.229
  max_depth=None: train R^2=0.837, test R^2=-0.298


In [10]:
# Binary target: is this book "highly rated" (>= 4.0)?
ml_df["highly_rated"] = (ml_df["avg_rating"] >= 4.0).astype(int)
print(ml_df["highly_rated"].value_counts())

y_class = ml_df["highly_rated"]
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_class, test_size=0.2, random_state=42, stratify=y_class
)
print("Train:", X_train_c.shape, "Test:", X_test_c.shape)

highly_rated
1    132
0     51
Name: count, dtype: int64
Train: (146, 11) Test: (37, 11)


In [11]:
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train_c, y_train_c)
logreg_preds = logreg.predict(X_test_c)

print("Logistic Regression:")
print("  Accuracy:", accuracy_score(y_test_c, logreg_preds))
print("  Precision:", precision_score(y_test_c, logreg_preds))
print("  Recall:", recall_score(y_test_c, logreg_preds))
print("  Confusion matrix:\n", confusion_matrix(y_test_c, logreg_preds))

Logistic Regression:
  Accuracy: 0.6756756756756757
  Precision: 0.7142857142857143
  Recall: 0.9259259259259259
  Confusion matrix:
 [[ 0 10]
 [ 2 25]]


C:\Users\QUEEN\OneDrive\Desktop\books-project\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [12]:
rfc = RandomForestClassifier(max_depth=6, n_estimators=200, random_state=42)
rfc.fit(X_train_c, y_train_c)
rfc_preds = rfc.predict(X_test_c)

print("Random Forest Classifier:")
print("  Accuracy:", accuracy_score(y_test_c, rfc_preds))
print("  Precision:", precision_score(y_test_c, rfc_preds))
print("  Recall:", recall_score(y_test_c, rfc_preds))
print("  Confusion matrix:\n", confusion_matrix(y_test_c, rfc_preds))

Random Forest Classifier:
  Accuracy: 0.7027027027027027
  Precision: 0.7222222222222222
  Recall: 0.9629629629629629
  Confusion matrix:
 [[ 0 10]
 [ 1 26]]


In [13]:
print(ml_df["highly_rated"].value_counts())

highly_rated
1    132
0     51
Name: count, dtype: int64


In [14]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_c_scaled = scaler.fit_transform(X_train_c)
X_test_c_scaled = scaler.transform(X_test_c)

In [15]:
logreg_bal = LogisticRegression(max_iter=1000, class_weight="balanced")
logreg_bal.fit(X_train_c_scaled, y_train_c)
logreg_bal_preds = logreg_bal.predict(X_test_c_scaled)

print("Logistic Regression (balanced):")
print("  Accuracy:", accuracy_score(y_test_c, logreg_bal_preds))
print("  Precision:", precision_score(y_test_c, logreg_bal_preds))
print("  Recall:", recall_score(y_test_c, logreg_bal_preds))
print("  Confusion matrix:\n", confusion_matrix(y_test_c, logreg_bal_preds))

Logistic Regression (balanced):
  Accuracy: 0.5675675675675675
  Precision: 0.7894736842105263
  Recall: 0.5555555555555556
  Confusion matrix:
 [[ 6  4]
 [12 15]]


In [16]:
rfc_bal = RandomForestClassifier(max_depth=6, n_estimators=200, random_state=42, class_weight="balanced")
rfc_bal.fit(X_train_c, y_train_c)
rfc_bal_preds = rfc_bal.predict(X_test_c)

print("Random Forest Classifier (balanced):")
print("  Accuracy:", accuracy_score(y_test_c, rfc_bal_preds))
print("  Precision:", precision_score(y_test_c, rfc_bal_preds))
print("  Recall:", recall_score(y_test_c, rfc_bal_preds))
print("  Confusion matrix:\n", confusion_matrix(y_test_c, rfc_bal_preds))

Random Forest Classifier (balanced):
  Accuracy: 0.6216216216216216
  Precision: 0.8095238095238095
  Recall: 0.6296296296296297
  Confusion matrix:
 [[ 6  4]
 [10 17]]


In [17]:
print("Overfitting check (Random Forest Classifier, different max_depth values):\n")
for depth in [2, 6, None]:
    model = RandomForestClassifier(max_depth=depth, n_estimators=200, random_state=42, class_weight="balanced")
    model.fit(X_train_c, y_train_c)
    train_acc = accuracy_score(y_train_c, model.predict(X_train_c))
    test_acc = accuracy_score(y_test_c, model.predict(X_test_c))
    print(f"  max_depth={depth}: train accuracy={train_acc:.3f}, test accuracy={test_acc:.3f}")

Overfitting check (Random Forest Classifier, different max_depth values):

  max_depth=2: train accuracy=0.630, test accuracy=0.541
  max_depth=6: train accuracy=0.801, test accuracy=0.622
  max_depth=None: train accuracy=1.000, test accuracy=0.649


In [18]:
# Export clean data for Tableau
df.to_csv("books_for_tableau.csv", index=False)
print("Exported", df.shape)

Exported (1940, 9)


In [19]:
results_df = pd.DataFrame({'actual': y_test.values, 'predicted': rf_preds})
results_df.to_csv('model_predictions.csv', index=False)
print(results_df.head())

   actual  predicted
0     4.0   3.577504
1     5.0   2.941947
2     1.0   4.753359
3     3.0   3.990150
4     4.0   4.246678
